In [2]:
##############################################################################
# SETUP: imports, demo prompt, model path registry, and helper functions
#
# This cell mirrors the pipeline in:
#   simulation/sample_normal_agent.py  ->  simulation/sot_env.py  ->  core/LM_hf.py
#                                          core/prompt_formatting.py
#                                          core/model_loader.py
#
# The pipeline works as follows:
#   1. SotopiaEnv builds a raw text prompt (intro + dialog history)
#   2. format_sotopia_prompt() wraps it in the model's chat template
#   3. The tokenizer encodes the formatted prompt into token IDs
#   4. model.generate() produces output token IDs (prompt + new tokens)
#   5. The output is decoded, the assistant response is extracted by
#      splitting on the model-specific delimiter, and special tokens
#      (</s>, <|im_end|>, <|eot_id|>, etc.) are stripped
#   6. For thinking models (DeepSeek-R1, Qwen3), <think>...</think>
#      blocks are extracted and removed from the response
#   7. The cleaned response (first line only) is appended to the dialog
#      history, and the next turn's prompt is built for the other model
##############################################################################

import torch
import re
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

# ── The demo prompt: first row of the OLMo x DeepSeek prompt_records CSV ──
# This is the raw prompt that SotopiaEnv.get_current_prompt() returns at turn 0.
# It's the intro (agent identity, scenario, goals) + "Turn 0: Mia Sanders said:"
DEMO_PROMPT = (
    "You are Mia Sanders. Respond only with short, direct dialogue.\n"
    "Scenario: Two people are in a room with a red box and a blue box. "
    "A key has been placed in one of the boxes.\n"
    "Participants: Mia Sanders and Ava Thompson\n"
    "Mia Sanders's goal: The other person is looking for the key. You know "
    "where it is. Guide them to the correct location. Be direct and concise. "
    "(Extra information: The key is in the red box.)\n"
    "Ava Thompson's goal: Unknown"
    "Respond only with dialogue. Do not narrate actions or describe thoughts. "
    "Do not say LEAVE. Keep your responses concise.\n"
    "Turn 0: Mia Sanders said:"
)

# ── Model path registry (from model_paths.json) ──
# SotopiaEnv reads this at __init__ and passes it to getLM()
MODEL_PATHS = {
    "allenai_OLMo-3-7B-Instruct": "/juice6/u/jshe/nlp/hf_models/allenai_OLMo-3-7B-Instruct",
    "DeepSeek-R1-Distill-Llama-8B": "/juice6/u/jshe/nlp/hf_models/DeepSeek-R1-Distill-Llama-8B",
    "google_gemma-7b-it": "/juice6/u/jshe/nlp/hf_models/google_gemma-7b-it",
    "Meta-Llama-3-8B-Instruct": "/juice6/u/jshe/nlp/hf_models/Meta-Llama-3-8B-Instruct",
    "Mistral-7B-Instruct-v0.2": "/juice6/u/jshe/nlp/hf_models/Mistral-7B-Instruct-v0.2",
    "Mistral-7B-Instruct-v0.3": "/juice6/u/jshe/nlp/hf_models/Mistral-7B-Instruct-v0.3",
    "Qwen2.5-7B-Instruct": "/juice6/u/jshe/nlp/hf_models/Qwen2.5-7B-Instruct",
    "Qwen3-8B": "/juice6/u/jshe/nlp/hf_models/Qwen3-8B",
    "Qwen_Qwen3-14B": "/juice6/u/jshe/nlp/hf_models/Qwen_Qwen3-14B",
}

# ── Generation settings (from sample_normal_agent.py) ──
TEMPERATURE = 0.7
MAX_NEW_TOKENS_NORMAL = 300    # for non-thinking models
MAX_NEW_TOKENS_THINKING = 4028 # for DeepSeek-R1, Qwen3-8B, Qwen3-14B
SEED = 0

print("Setup complete. Demo prompt length:", len(DEMO_PROMPT), "chars")
print(f"Demo prompt preview:\n{DEMO_PROMPT[:200]}...")

Setup complete. Demo prompt length: 578 chars
Demo prompt preview:
You are Mia Sanders. Respond only with short, direct dialogue.
Scenario: Two people are in a room with a red box and a blue box. A key has been placed in one of the boxes.
Participants: Mia Sanders an...


In [ ]:
##############################################################################
# HELPER FUNCTIONS
#
# These replicate the exact logic from the pipeline so you can see each step.
# In production, these live in:
#   core/prompt_formatting.py  -> format_sotopia_prompt()
#   core/LM_hf.py             -> BaseLM.get_response_format()
#                              -> BaseLM.get_result_from_output()
##############################################################################

def format_prompt(model_name, raw_prompt, tokenizer):
    """Wrap the raw Sotopia prompt in the model's chat template.
    
    This is what core/prompt_formatting.py:format_sotopia_prompt() does.
    Each model family uses a different chat template format:
      - Mistral v0.2/v0.3:  [INST]...[/INST]
      - Llama 3 Instruct:   <|begin_of_text|><|start_header_id|>system...
      - Gemma:              <start_of_turn>user\n...<end_of_turn>\n<start_of_turn>model\n
      - Qwen/OLMo (ChatML): <|im_start|>user\n...<|im_end|>\n<|im_start|>assistant\n
      - DeepSeek-R1:        <｜begin▁of▁sentence｜><｜User｜>...<｜Assistant｜>
    
    The pipeline explicitly skips tokenizer.apply_chat_template() for OLMo
    (it's in _skip_auto_template) and uses manual ChatML formatting instead.
    For other models, it tries apply_chat_template() first, then falls back
    to manual formatting.
    """
    # Models that skip the tokenizer's built-in chat_template
    _skip_auto_template = {"allenai_OLMo-3-7B-Instruct"}
    
    # Try tokenizer.apply_chat_template first (unless skipped)
    ct = getattr(tokenizer, "chat_template", None)
    if ct and model_name not in _skip_auto_template:
        try:
            return tokenizer.apply_chat_template(
                [{"role": "user", "content": raw_prompt}],
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            pass
    
    # Manual fallbacks for each model family
    if model_name in ("Mistral-7B-Instruct-v0.2", "Mistral-7B-Instruct-v0.3"):
        return f"[INST]{raw_prompt}[/INST]"
    
    if "Llama" in model_name and "Instruct" in model_name:
        return (
            "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            "You are a helpful assistant<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n\n"
            f"{raw_prompt}<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
        )
    
    if model_name in ("gemma-2-9b-it", "google_gemma-7b-it"):
        return f"<start_of_turn>user\n{raw_prompt}<end_of_turn>\n<start_of_turn>model\n"
    
    if model_name in ("Qwen2.5-7B-Instruct", "Qwen_Qwen3-14B", "Qwen3-8B",
                       "allenai_OLMo-3-7B-Instruct"):
        return f"<|im_start|>user\n{raw_prompt}<|im_end|>\n<|im_start|>assistant\n"
    
    if model_name == "DeepSeek-R1-Distill-Llama-8B":
        return f"<｜begin▁of▁sentence｜><｜User｜>{raw_prompt}<｜Assistant｜>"
    
    return f"[INST]{raw_prompt}[/INST]"


def extract_response(model_name, decoded_text):
    """Extract the assistant's response from the full decoded output.
    
    This is what core/LM_hf.py:BaseLM.get_response_format() does.
    After model.generate() produces token IDs for [prompt + response],
    we decode the entire sequence back to text, then split on the
    model-specific delimiter to isolate just the assistant's reply.
    """
    if model_name in ("Mistral-7B-Instruct-v0.2", "Mistral-7B-Instruct-v0.3"):
        return decoded_text.split('[/INST]')[-1]
    elif model_name == "DeepSeek-R1-Distill-Llama-8B":
        return decoded_text.split('<｜Assistant｜>')[-1]
    elif "Llama" in model_name and "Instruct" in model_name:
        return decoded_text.split('<|end_header_id|>')[-1].replace('<|eot_id|>', '')
    elif model_name in ("gemma-2-9b-it", "google_gemma-7b-it"):
        return decoded_text.split('<start_of_turn>model')[-1]
    elif model_name in ("Qwen2.5-7B-Instruct", "Qwen_Qwen3-14B", "Qwen3-8B",
                         "allenai_OLMo-3-7B-Instruct"):
        return decoded_text.split('<|im_start|>assistant\n')[-1]
    return decoded_text


def strip_special_tokens(text):
    """Remove leftover special tokens from the extracted response.
    
    This is what core/LM_hf.py:BaseLM.get_result_from_output() does
    AFTER extract_response(). Different models leave different EOS/padding
    tokens in the decoded text; we strip all known ones.
    """
    for tok in ['</s>', '<|im_end|>', '<|eot_id|>',
                '<｜end▁of▁sentence｜>', '<｜begin▁of▁sentence｜>',
                '<|endoftext|>', '<eos>']:
        text = text.replace(tok, '')
    return text


def extract_thinking(text):
    """Extract and remove <think>...</think> blocks from thinking models.
    
    DeepSeek-R1 and Qwen3 models produce chain-of-thought inside <think> tags
    before their actual response. The pipeline (core/LM_hf.py lines 71-80)
    extracts this trace for logging, then removes it from the response.
    """
    think_match = re.search(r'<think>(.*?)</think>', text, flags=re.DOTALL)
    if think_match:
        thinking_trace = think_match.group(1).strip()
        text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    elif '<think>' in text:
        # Model ran out of max_new_tokens mid-thinking (no closing </think>)
        thinking_trace = text.split('<think>', 1)[1].strip()
        text = text.split('<think>', 1)[0]
    else:
        thinking_trace = ""
    return text.strip(), thinking_trace


def full_pipeline(model_name, model, tokenizer, raw_prompt, max_new_tokens):
    """Run the complete generate-decode-strip pipeline for one model.
    
    This mirrors what happens in core/LM_hf.py:LM_nnsight.generate_response():
      1. format_prompt()       <- core/prompt_formatting.py
      2. tokenizer.encode()    <- tokenization
      3. model.generate()      <- generation
      4. tokenizer.decode()    <- decoding
      5. extract_response()    <- split on delimiter
      6. strip_special_tokens() <- remove EOS etc.
      7. extract_thinking()    <- remove <think> blocks
      8. .split('\n')[0]       <- take first line only (sot_env.py line 424)
    
    In the actual pipeline, nnsight wraps model.generate(), but the
    tokenization and post-processing are identical.
    """
    # Step 1: Wrap raw prompt in chat template
    formatted = format_prompt(model_name, raw_prompt, tokenizer)
    print("=" * 70)
    print(f"STEP 1 — Chat-template formatted prompt ({len(formatted)} chars):")
    print(repr(formatted[:500]))
    print()
    
    # Step 2: Tokenize
    input_ids = tokenizer.encode(formatted, return_tensors="pt").to(model.device)
    print(f"STEP 2 — Tokenized: {input_ids.shape[1]} tokens")
    print(f"  First 20 token IDs: {input_ids[0, :20].tolist()}")
    print(f"  Last 10 token IDs:  {input_ids[0, -10:].tolist()}")
    # Decode individual tokens so you can see the vocabulary
    print(f"  First 20 tokens decoded: {[tokenizer.decode([t]) for t in input_ids[0, :20].tolist()]}")
    print()
    
    # Step 3: Generate
    with torch.no_grad():
        output_ids = model.generate(input_ids, max_new_tokens=max_new_tokens)
    new_token_ids = output_ids[0, input_ids.shape[1]:]  # only the generated part
    print(f"STEP 3 — Generated {len(new_token_ids)} new tokens")
    print(f"  New token IDs: {new_token_ids.tolist()[:30]}{'...' if len(new_token_ids) > 30 else ''}")
    print(f"  New tokens decoded: {[tokenizer.decode([t]) for t in new_token_ids.tolist()[:30]]}")
    print()
    
    # Step 4: Decode full output (prompt + generated) back to text
    # The pipeline decodes the ENTIRE sequence, not just new tokens
    full_decoded = tokenizer.decode(output_ids[0])
    print(f"STEP 4 — Full decoded text ({len(full_decoded)} chars):")
    print(repr(full_decoded[-300:]))
    print()
    
    # Step 5: Extract assistant response by splitting on delimiter
    response = extract_response(model_name, full_decoded)
    print(f"STEP 5 — After extract_response (split on delimiter):")
    print(repr(response[:300]))
    print()
    
    # Step 6: Strip special tokens
    response = strip_special_tokens(response)
    print(f"STEP 6 — After strip_special_tokens:")
    print(repr(response[:300]))
    print()
    
    # Step 7: Extract thinking trace (for DeepSeek-R1, Qwen3)
    response, thinking = extract_thinking(response)
    if thinking:
        print(f"STEP 7 — Thinking trace extracted ({len(thinking)} chars):")
        print(f"  {thinking[:200]}...")
    else:
        print(f"STEP 7 — No thinking trace (non-thinking model)")
    print()
    
    # Step 8: Take first line only (sot_env.py:_act() line 424)
    # Then strip "Name: " prefix if present (line 428-429)
    first_line = response.split('\n')[0]
    if ': ' in first_line:
        first_line = ''.join(first_line.split(': ')[1:])
    print(f"STEP 8 — Final result (first line, prefix stripped):")
    print(f"  '{first_line}'")
    print()
    print(f"This text gets appended to the dialog history as:")
    print(f"  'Turn 0: Mia Sanders said:{first_line}'")
    print(f"Then the next turn's prompt is built for the OTHER model")
    print(f"with that line included in the conversation history.")
    print("=" * 70)
    
    return first_line

print("Helper functions defined.")

In [ ]:
##############################################################################
# MODEL 1: allenai_OLMo-3-7B-Instruct
#
# Architecture: OLMo (hf_olmo). Requires `import hf_olmo` to register
#   the auto-map classes before AutoModelForCausalLM can load it.
# Chat template: ChatML format — <|im_start|>user\n...<|im_end|>\n<|im_start|>assistant\n
# Response delimiter: split on '<|im_start|>assistant\n'
# Special tokens to strip: <|im_end|>
# Thinking model: NO (max_new_tokens = 300)
#
# NOTE: The pipeline SKIPS tokenizer.apply_chat_template() for OLMo
#   (it's in _skip_auto_template in prompt_formatting.py) and manually
#   constructs the ChatML string. This is because OLMo's tokenizer
#   may produce a slightly different template than what the pipeline expects.
#
# In the real pipeline (core/model_loader.py lines 335-348), OLMo is loaded
# via AutoModelForCausalLM with hf_olmo registered, then wrapped in
# _OLMoCausalLMCompat to expose model.model.layers (pointing to
# olmo.model.transformer.blocks). Here we load it directly since we
# only need generate(), not nnsight layer access.
##############################################################################

import hf_olmo  # must import before AutoModelForCausalLM to register OLMo classes

MODEL_NAME = "allenai_OLMo-3-7B-Instruct"
MODEL_PATH = MODEL_PATHS[MODEL_NAME]

print(f"Loading {MODEL_NAME} from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

# Add pad token if missing (model_loader.py lines 367-369)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True
)

if tokenizer.pad_token_id is not None:
    model.resize_token_embeddings(len(tokenizer))

# Set generation config (model_loader.py lines 376-390)
try:
    model.generation_config = GenerationConfig.from_pretrained(MODEL_PATH)
except Exception:
    pass
model.generation_config.do_sample = True
model.generation_config.temperature = TEMPERATURE
model.to("cuda").eval()

print(f"Loaded. Vocab size: {len(tokenizer)}, pad_token: {tokenizer.pad_token!r}")
print(f"Special tokens: {tokenizer.all_special_tokens}")
print()

result = full_pipeline(MODEL_NAME, model, tokenizer, DEMO_PROMPT, MAX_NEW_TOKENS_NORMAL)

In [ ]:
# Free GPU memory before loading the next model
del model, tokenizer
torch.cuda.empty_cache()
print("GPU memory freed.")

In [ ]:
##############################################################################
# MODEL 2: DeepSeek-R1-Distill-Llama-8B
#
# Architecture: standard (AutoModelForCausalLM) — it's a Llama architecture
#   distilled from DeepSeek-R1, so it loads like any Llama model.
# Chat template: DeepSeek's custom format with fullwidth Unicode delimiters:
#   <｜begin▁of▁sentence｜><｜User｜>{prompt}<｜Assistant｜>
#   Note: these are NOT regular ASCII pipes/underscores — they are
#   fullwidth characters (U+FF5C, U+2581) that tokenize as single tokens.
# Response delimiter: split on '<｜Assistant｜>'
# Special tokens to strip: <｜end▁of▁sentence｜>, <｜begin▁of▁sentence｜>
# Thinking model: YES (max_new_tokens = 1024)
#   Produces <think>chain-of-thought</think> before the actual response.
#   The pipeline extracts the thinking trace and removes it.
#
# This is one of the most interesting models to inspect because:
#   1. The unusual Unicode special tokens
#   2. The <think>...</think> chain-of-thought that gets stripped
#   3. If max_new_tokens is too low, it may never finish thinking
##############################################################################

MODEL_NAME = "DeepSeek-R1-Distill-Llama-8B"
MODEL_PATH = MODEL_PATHS[MODEL_NAME]

print(f"Loading {MODEL_NAME} from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True
)
model.resize_token_embeddings(len(tokenizer))

try:
    model.generation_config = GenerationConfig.from_pretrained(MODEL_PATH)
except Exception:
    pass
model.generation_config.do_sample = True
model.generation_config.temperature = TEMPERATURE
model.to("cuda").eval()

print(f"Loaded. Vocab size: {len(tokenizer)}, pad_token: {tokenizer.pad_token!r}")
# Show DeepSeek's unusual special tokens
print(f"Special tokens: {tokenizer.all_special_tokens}")
print(f"EOS token: {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print()

# DeepSeek-R1 is a THINKING model — it gets 1024 max_new_tokens
result = full_pipeline(MODEL_NAME, model, tokenizer, DEMO_PROMPT, MAX_NEW_TOKENS_THINKING)

In [ ]:
del model, tokenizer
torch.cuda.empty_cache()
print("GPU memory freed.")

In [ ]:
##############################################################################
# MODEL 3: google_gemma-7b-it
#
# Architecture: standard (AutoModelForCausalLM).
# Chat template: Gemma format —
#   <start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n
# Response delimiter: split on '<start_of_turn>model'
# Special tokens to strip: <eos>
# Thinking model: NO (max_new_tokens = 300)
#
# Gemma uses <start_of_turn>/<end_of_turn> markers instead of the more
# common [INST] or ChatML format. The pipeline hardcodes this in
# prompt_formatting.py for "google_gemma-7b-it".
##############################################################################

MODEL_NAME = "google_gemma-7b-it"
MODEL_PATH = MODEL_PATHS[MODEL_NAME]

print(f"Loading {MODEL_NAME} from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True
)
model.resize_token_embeddings(len(tokenizer))

try:
    model.generation_config = GenerationConfig.from_pretrained(MODEL_PATH)
except Exception:
    pass
model.generation_config.do_sample = True
model.generation_config.temperature = TEMPERATURE
model.to("cuda").eval()

print(f"Loaded. Vocab size: {len(tokenizer)}, pad_token: {tokenizer.pad_token!r}")
print(f"Special tokens: {tokenizer.all_special_tokens}")
print()

result = full_pipeline(MODEL_NAME, model, tokenizer, DEMO_PROMPT, MAX_NEW_TOKENS_NORMAL)

In [ ]:
del model, tokenizer
torch.cuda.empty_cache()
print("GPU memory freed.")

In [ ]:
##############################################################################
# MODEL 4: Meta-Llama-3-8B-Instruct
#
# Architecture: standard (AutoModelForCausalLM).
# Chat template: Llama 3 Instruct format —
#   <|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n
#   You are a helpful assistant<|eot_id|>
#   <|start_header_id|>user<|end_header_id|>\n\n
#   {prompt}<|eot_id|>
#   <|start_header_id|>assistant<|end_header_id|>\n\n
#
# Response delimiter: split on '<|end_header_id|>' (takes last segment),
#   then also remove any '<|eot_id|>' within it.
# Special tokens to strip: <|eot_id|>
# Thinking model: NO (max_new_tokens = 300)
#
# Llama 3's template is notably different from Llama 2 (which used [INST]).
# The pipeline checks for "Llama" + "Instruct" in the name to pick this
# template, vs "Llama" + "Chat" for the Llama 2 [INST] format.
# Note: the pipeline adds a system message "You are a helpful assistant"
# even though the Sotopia prompt already contains full persona instructions.
##############################################################################

MODEL_NAME = "Meta-Llama-3-8B-Instruct"
MODEL_PATH = MODEL_PATHS[MODEL_NAME]

print(f"Loading {MODEL_NAME} from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True
)
model.resize_token_embeddings(len(tokenizer))

try:
    model.generation_config = GenerationConfig.from_pretrained(MODEL_PATH)
except Exception:
    pass
model.generation_config.do_sample = True
model.generation_config.temperature = TEMPERATURE
model.to("cuda").eval()

print(f"Loaded. Vocab size: {len(tokenizer)}, pad_token: {tokenizer.pad_token!r}")
print(f"Special tokens: {tokenizer.all_special_tokens}")
print()

result = full_pipeline(MODEL_NAME, model, tokenizer, DEMO_PROMPT, MAX_NEW_TOKENS_NORMAL)

In [ ]:
del model, tokenizer
torch.cuda.empty_cache()
print("GPU memory freed.")

In [ ]:
##############################################################################
# MODEL 5: Mistral-7B-Instruct-v0.2
#
# Architecture: standard (AutoModelForCausalLM).
# Chat template: Mistral Instruct format — [INST]{prompt}[/INST]
#   This is the simplest template in the study. No system message,
#   no special begin/end tokens beyond the [INST] markers.
# Response delimiter: split on '[/INST]' (takes last segment)
# Special tokens to strip: </s>
# Thinking model: NO (max_new_tokens = 300)
#
# Note: prompt_formatting.py hardcodes the [INST] format for Mistral v0.1/v0.2/v0.3
# and does NOT attempt tokenizer.apply_chat_template() because it hits the
# manual fallback first. However, for v0.2 specifically, the tokenizer's
# built-in chat_template (if present) would produce the same format.
#
# Mistral v0.2 and v0.3 share the same template format but may have
# different vocabularies and model weights.
##############################################################################

MODEL_NAME = "Mistral-7B-Instruct-v0.2"
MODEL_PATH = MODEL_PATHS[MODEL_NAME]

print(f"Loading {MODEL_NAME} from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True
)
model.resize_token_embeddings(len(tokenizer))

try:
    model.generation_config = GenerationConfig.from_pretrained(MODEL_PATH)
except Exception:
    pass
model.generation_config.do_sample = True
model.generation_config.temperature = TEMPERATURE
model.to("cuda").eval()

print(f"Loaded. Vocab size: {len(tokenizer)}, pad_token: {tokenizer.pad_token!r}")
print(f"Special tokens: {tokenizer.all_special_tokens}")
print()

result = full_pipeline(MODEL_NAME, model, tokenizer, DEMO_PROMPT, MAX_NEW_TOKENS_NORMAL)

In [ ]:
del model, tokenizer
torch.cuda.empty_cache()
print("GPU memory freed.")

In [ ]:
##############################################################################
# MODEL 6: Mistral-7B-Instruct-v0.3
#
# Architecture: standard (AutoModelForCausalLM).
# Chat template: Same as v0.2 — [INST]{prompt}[/INST]
# Response delimiter: split on '[/INST]'
# Special tokens to strip: </s>
# Thinking model: NO (max_new_tokens = 300)
#
# Identical template to v0.2 but different weights/vocabulary.
# The pipeline uses the exact same code path for both.
# Included separately so you can compare tokenization differences
# between v0.2 and v0.3 (e.g., different BPE merges, vocab size).
##############################################################################

MODEL_NAME = "Mistral-7B-Instruct-v0.3"
MODEL_PATH = MODEL_PATHS[MODEL_NAME]

print(f"Loading {MODEL_NAME} from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True
)
model.resize_token_embeddings(len(tokenizer))

try:
    model.generation_config = GenerationConfig.from_pretrained(MODEL_PATH)
except Exception:
    pass
model.generation_config.do_sample = True
model.generation_config.temperature = TEMPERATURE
model.to("cuda").eval()

print(f"Loaded. Vocab size: {len(tokenizer)}, pad_token: {tokenizer.pad_token!r}")
print(f"Special tokens: {tokenizer.all_special_tokens}")
print()

result = full_pipeline(MODEL_NAME, model, tokenizer, DEMO_PROMPT, MAX_NEW_TOKENS_NORMAL)

In [ ]:
del model, tokenizer
torch.cuda.empty_cache()
print("GPU memory freed.")

In [ ]:
##############################################################################
# MODEL 7: Qwen2.5-7B-Instruct
#
# Architecture: standard (AutoModelForCausalLM).
# Chat template: ChatML format —
#   <|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n
#   This is the same ChatML format used by OLMo-3, but Qwen's tokenizer
#   has <|im_start|> and <|im_end|> as native special tokens, whereas
#   OLMo encodes them differently.
# Response delimiter: split on '<|im_start|>assistant\n'
# Special tokens to strip: <|im_end|>, <|endoftext|>
# Thinking model: NO (max_new_tokens = 300)
#
# NOTE: Unlike Qwen3 models, Qwen2.5 does NOT produce <think> blocks.
# The pipeline treats it as a non-thinking model with 300 max tokens.
# Qwen2.5's tokenizer.apply_chat_template() would produce something
# slightly different (with a system prompt), but the pipeline's manual
# fallback is what actually runs (prompt_formatting.py lines 54-61).
##############################################################################

MODEL_NAME = "Qwen2.5-7B-Instruct"
MODEL_PATH = MODEL_PATHS[MODEL_NAME]

print(f"Loading {MODEL_NAME} from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True
)
model.resize_token_embeddings(len(tokenizer))

try:
    model.generation_config = GenerationConfig.from_pretrained(MODEL_PATH)
except Exception:
    pass
model.generation_config.do_sample = True
model.generation_config.temperature = TEMPERATURE
model.to("cuda").eval()

print(f"Loaded. Vocab size: {len(tokenizer)}, pad_token: {tokenizer.pad_token!r}")
print(f"Special tokens: {tokenizer.all_special_tokens}")
print()

result = full_pipeline(MODEL_NAME, model, tokenizer, DEMO_PROMPT, MAX_NEW_TOKENS_NORMAL)

In [ ]:
del model, tokenizer
torch.cuda.empty_cache()
print("GPU memory freed.")

In [ ]:
##############################################################################
# MODEL 8: Qwen3-8B
#
# Architecture: standard (AutoModelForCausalLM).
# Chat template: ChatML format (same as Qwen2.5 and OLMo) —
#   <|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n
# Response delimiter: split on '<|im_start|>assistant\n'
# Special tokens to strip: <|im_end|>, <|endoftext|>
# Thinking model: YES (max_new_tokens = 1024)
#   Like DeepSeek-R1, Qwen3 produces <think>...</think> chain-of-thought
#   before the actual response. The pipeline strips this in
#   get_result_from_output() (LM_hf.py lines 71-80).
#
# IMPORTANT: Qwen3-8B is in the THINKING_MODELS set in sample_normal_agent.py
# (line 27), so it gets max_new_tokens=1024 instead of 300. This matters
# because the <think> block can be very long. If it exceeds 1024 tokens,
# the model produces no actual response — the pipeline handles this by
# checking for an unclosed <think> tag (LM_hf.py lines 75-78).
##############################################################################

MODEL_NAME = "Qwen3-8B"
MODEL_PATH = MODEL_PATHS[MODEL_NAME]

print(f"Loading {MODEL_NAME} from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True
)
model.resize_token_embeddings(len(tokenizer))

try:
    model.generation_config = GenerationConfig.from_pretrained(MODEL_PATH)
except Exception:
    pass
model.generation_config.do_sample = True
model.generation_config.temperature = TEMPERATURE
model.to("cuda").eval()

print(f"Loaded. Vocab size: {len(tokenizer)}, pad_token: {tokenizer.pad_token!r}")
print(f"Special tokens: {tokenizer.all_special_tokens}")
print()

# Qwen3-8B is a THINKING model — it gets 1024 max_new_tokens
result = full_pipeline(MODEL_NAME, model, tokenizer, DEMO_PROMPT, MAX_NEW_TOKENS_THINKING)

In [ ]:
del model, tokenizer
torch.cuda.empty_cache()
print("GPU memory freed.")

In [ ]:
##############################################################################
# MODEL 9: Qwen_Qwen3-14B
#
# Architecture: standard (AutoModelForCausalLM).
#   Requires transformers>=4.51 (noted in model_loader.py line 8).
# Chat template: ChatML format (same as Qwen2.5, Qwen3-8B, OLMo) —
#   <|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n
# Response delimiter: split on '<|im_start|>assistant\n'
# Special tokens to strip: <|im_end|>, <|endoftext|>
# Thinking model: YES (max_new_tokens = 1024)
#   Like Qwen3-8B, produces <think>...</think> blocks.
#
# This is the largest model in the study at 14B parameters.
# It uses the same ChatML template as Qwen3-8B and Qwen2.5, but
# is listed in THINKING_MODELS in sample_normal_agent.py (line 27)
# as "Qwen_Qwen3-14B", so it gets 1024 max_new_tokens.
#
# WARNING: This model is ~28GB in fp16. Make sure you have enough
# GPU memory. If running on an 80GB GPU after clearing previous models,
# this should fit fine.
##############################################################################

MODEL_NAME = "Qwen_Qwen3-14B"
MODEL_PATH = MODEL_PATHS[MODEL_NAME]

print(f"Loading {MODEL_NAME} from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True
)
model.resize_token_embeddings(len(tokenizer))

try:
    model.generation_config = GenerationConfig.from_pretrained(MODEL_PATH)
except Exception:
    pass
model.generation_config.do_sample = True
model.generation_config.temperature = TEMPERATURE
model.to("cuda").eval()

print(f"Loaded. Vocab size: {len(tokenizer)}, pad_token: {tokenizer.pad_token!r}")
print(f"Special tokens: {tokenizer.all_special_tokens}")
print()

# Qwen3-14B is a THINKING model — it gets 1024 max_new_tokens
result = full_pipeline(MODEL_NAME, model, tokenizer, DEMO_PROMPT, MAX_NEW_TOKENS_THINKING)

In [ ]:
del model, tokenizer
torch.cuda.empty_cache()
print("GPU memory freed.")